# Traffic Demand Prediction - Improved Solution
OOF CV R2 Score: 0.972746 | Hackathon Score: 97.2746

In [2]:
"""Demand prediction â€” EDA + baseline + GBM. Optimize for RÂ²."""
import sys, time, json
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import OrdinalEncoder

DATA = "dataset\\"
TARGET = "demand"
RNG = 42

t0 = time.time()

train = pd.read_csv("dataset\\train.csv")
test = pd.read_csv("dataset\\test.csv")
print(f"train {train.shape}  test {test.shape}")
print("dtypes:\n", train.dtypes)

# ---------- EDA ----------
print("\n--- target ---")
print(train[TARGET].describe())
print("nulls in target:", train[TARGET].isna().sum())

print("\n--- missing per col (train) ---")
print(train.isna().sum().sort_values(ascending=False))

print("\n--- cardinalities ---")
for c in train.columns:
    print(f"  {c}: nunique={train[c].nunique()}")

print("\n--- day range ---")
print("train day:", sorted(train.day.unique())[:5], "...", sorted(train.day.unique())[-5:])
print("test  day:", sorted(test.day.unique())[:5], "...", sorted(test.day.unique())[-5:])

print("\n--- timestamp samples ---")
print(train.timestamp.value_counts().head())

# ---------- feature engineering ----------
def fe(df):
    df = df.copy()
    # timestamp like "0:0", "2:15" -> minute-of-day
    ts = df["timestamp"].str.split(":", expand=True).astype(int)
    df["minute"] = ts[0] * 60 + ts[1]
    df["hour"] = ts[0]
    df["minute_of_hour"] = ts[1]
    # cyclic encoding
    df["min_sin"] = np.sin(2 * np.pi * df["minute"] / 1440)
    df["min_cos"] = np.cos(2 * np.pi * df["minute"] / 1440)
    # day-of-week if day is a sequential day index
    df["dow"] = df["day"] % 7
    # geohash prefixes (spatial granularity)
    for n in (3, 4, 5):
        df[f"gh{n}"] = df["geohash"].str[:n]
    return df

train_fe = fe(train)
test_fe  = fe(test)

# geohash is high-cardinality (1241) â€” use target encoding instead of native categorical.
# Other categoricals (gh3/gh4/gh5/RoadType/...) stay ordinal-encoded categoricals.
high_card_cols = ["geohash"]  # target-encode
cat_cols = ["gh3", "gh4", "gh5", "RoadType", "LargeVehicles", "Landmarks", "Weather"]
num_cols = ["day", "minute", "hour", "minute_of_hour", "min_sin", "min_cos", "dow",
            "NumberofLanes", "Temperature"]

for c in cat_cols + high_card_cols:
    train_fe[c] = train_fe[c].fillna("MISSING").astype(str)
    test_fe[c]  = test_fe[c].fillna("MISSING").astype(str)
temp_med = train_fe["Temperature"].median()
train_fe["Temperature"] = train_fe["Temperature"].fillna(temp_med)
test_fe["Temperature"]  = test_fe["Temperature"].fillna(temp_med)

# Out-of-fold target encoding for geohash (with smoothing)
def target_encode_oof(train_df, test_df, col, y, n_splits=5, smoothing=20, seed=RNG):
    global_mean = y.mean()
    oof = np.zeros(len(train_df), dtype=np.float32)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in kf.split(train_df):
        agg = pd.DataFrame({col: train_df[col].iloc[tr].values, "y": y[tr]}) \
                .groupby(col)["y"].agg(["mean", "count"])
        smoothed = (agg["mean"] * agg["count"] + global_mean * smoothing) / (agg["count"] + smoothing)
        mp = smoothed.to_dict()
        oof[va] = train_df[col].iloc[va].map(mp).fillna(global_mean).values
    # test encoding uses full train
    agg = pd.DataFrame({col: train_df[col].values, "y": y}).groupby(col)["y"].agg(["mean", "count"])
    smoothed = (agg["mean"] * agg["count"] + global_mean * smoothing) / (agg["count"] + smoothing)
    mp = smoothed.to_dict()
    test_enc = test_df[col].map(mp).fillna(global_mean).values.astype(np.float32)
    return oof, test_enc

y_raw = train_fe[TARGET].astype(np.float32).values
for c in high_card_cols:
    oof, te = target_encode_oof(train_fe, test_fe, c, y_raw)
    train_fe[f"{c}_te"] = oof
    test_fe[f"{c}_te"]  = te

# ordinal encode remaining categoricals
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
all_cat = pd.concat([train_fe[cat_cols], test_fe[cat_cols]], axis=0)
enc.fit(all_cat)
train_fe[cat_cols] = enc.transform(train_fe[cat_cols])
test_fe[cat_cols]  = enc.transform(test_fe[cat_cols])

te_cols = [f"{c}_te" for c in high_card_cols]
features = cat_cols + num_cols + te_cols

X = train_fe[features].astype(np.float32)
y = y_raw
X_test = test_fe[features].astype(np.float32)

cat_idx = [features.index(c) for c in cat_cols]

print(f"\nfeatures: {len(features)}  cat: {len(cat_cols)}  num: {len(num_cols)}")
print(f"X={X.shape}  X_test={X_test.shape}")
print(f"elapsed: {time.time()-t0:.1f}s")

# ---------- CV ----------
def cv_score(model_factory, name, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RNG)
    scores, oof = [], np.zeros(len(y), dtype=np.float32)
    for fold, (tr, va) in enumerate(kf.split(X)):
        m = model_factory()
        m.fit(X.iloc[tr], y[tr])
        p = m.predict(X.iloc[va])
        oof[va] = p
        s = r2_score(y[va], p)
        scores.append(s)
        print(f"  [{name}] fold {fold+1}: RÂ²={s:.5f}")
    mean, std = np.mean(scores), np.std(scores)
    print(f"  [{name}] mean RÂ²={mean:.5f} Â± {std:.5f}  (OOF RÂ²={r2_score(y, oof):.5f})")
    return mean, std, oof

print("\n=== Ridge baseline ===")
cv_score(lambda: Ridge(alpha=1.0, random_state=RNG), "ridge", n_splits=5)

print("\n=== HistGradientBoosting (default) ===")
cv_score(lambda: HistGradientBoostingRegressor(
    max_iter=500, learning_rate=0.05, max_depth=None, max_leaf_nodes=63,
    min_samples_leaf=20, l2_regularization=1.0,
    categorical_features=cat_idx, random_state=RNG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=30),
    "hgb", n_splits=5)

# ---------- train final on full train and predict ----------
print("\n=== final HGB on full train ===")
final = HistGradientBoostingRegressor(
    max_iter=1500, learning_rate=0.03, max_leaf_nodes=127,
    min_samples_leaf=20, l2_regularization=1.0,
    categorical_features=cat_idx, random_state=RNG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=50)
final.fit(X, y)
print(f"  n_iter trained: {final.n_iter_}")

pred = final.predict(X_test)
pred = np.clip(pred, 0, None)  # demand >= 0

sub = pd.DataFrame({"Index": test_fe["Index"].values, "demand": pred})
sub.to_csv(f"{DATA}/submission.csv", index=False)
print(f"\nsubmission saved: {DATA}/submission.csv  rows={len(sub)}")
print(sub.head())

print(f"\ntotal elapsed: {time.time()-t0:.1f}s")

train (77299, 11)  test (41778, 10)
dtypes:
 Index              int64
geohash              str
day                int64
timestamp            str
demand           float64
RoadType             str
NumberofLanes      int64
LargeVehicles        str
Landmarks            str
Temperature      float64
Weather              str
dtype: object

--- target ---
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64
nulls in target: 0

--- missing per col (train) ---
Temperature      2495
Weather           797
RoadType          600
day                 0
geohash             0
Index               0
timestamp           0
NumberofLanes       0
demand              0
Landmarks           0
LargeVehicles       0
dtype: int64

--- cardinalities ---
  Index: nunique=77299
  geohash: nunique=1249
  day: nunique=2
  timestamp: nunique=96
  demand: nunique=76715
  R

c:\Users\nanin\OneDrive\Desktop\FlipkartGrid_Hackathon\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.510066873939181e-11.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\nanin\OneDrive\Desktop\FlipkartGrid_Hackathon\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 8.473907686656901e-11.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


  [ridge] fold 5: RÂ²=0.76566
  [ridge] mean RÂ²=0.76338 Â± 0.00892  (OOF RÂ²=0.76355)

=== HistGradientBoosting (default) ===
  [hgb] fold 1: RÂ²=0.94162
  [hgb] fold 2: RÂ²=0.94091
  [hgb] fold 3: RÂ²=0.93666
  [hgb] fold 4: RÂ²=0.93766
  [hgb] fold 5: RÂ²=0.94008
  [hgb] mean RÂ²=0.93939 Â± 0.00191  (OOF RÂ²=0.93942)

=== final HGB on full train ===
  n_iter trained: 554

submission saved: dataset\/submission.csv  rows=41778
   Index    demand
0      0  0.037238
1      1  0.029544
2      2  0.021984
3      3  0.032384
4      4  0.035531

total elapsed: 57.8s


In [3]:
"""Demand prediction v2 â€” add day-48 lag + geohash aggregates. Target RÂ²â‰ 0.99."""
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import OrdinalEncoder

DATA = "dataset\\"
TARGET = "demand"
RNG = 42
t0 = time.time()

train = pd.read_csv(f"{DATA}train.csv")
test  = pd.read_csv(f"{DATA}test.csv")
print(f"train {train.shape}  test {test.shape}")

# ---------- timestamp / spatial FE ----------
def fe(df):
    df = df.copy()
    ts = df["timestamp"].str.split(":", expand=True).astype(int)
    df["minute"] = ts[0]*60 + ts[1]
    df["hour"]   = ts[0]
    df["min_sin"] = np.sin(2*np.pi*df["minute"]/1440)
    df["min_cos"] = np.cos(2*np.pi*df["minute"]/1440)
    df["dow"]    = df["day"] % 7
    for n in (3, 4, 5):
        df[f"gh{n}"] = df["geohash"].str[:n]
    return df

train_fe = fe(train)
test_fe  = fe(test)

# ---------- day-48 lag features ----------
# day-48 universe (every train row where day==48)
d48 = train_fe[train_fe["day"] == 48][["geohash", "hour", "minute", TARGET]].copy()

# (geohash, minute) exact same-timestamp lag
lag_exact = d48.groupby(["geohash", "minute"])[TARGET].mean().rename("lag_same_ts").reset_index()
# (geohash, hour) coarser lag
lag_hour  = d48.groupby(["geohash", "hour"])[TARGET].mean().rename("lag_same_hour").reset_index()
# per-geohash day-48 aggregates
gh_aggs = d48.groupby("geohash")[TARGET].agg(
    gh_d48_mean="mean", gh_d48_std="std", gh_d48_med="median",
    gh_d48_min="min", gh_d48_max="max", gh_d48_q25=lambda s: s.quantile(0.25),
    gh_d48_q75=lambda s: s.quantile(0.75), gh_d48_n="count"
).reset_index()
gh_aggs["gh_d48_std"] = gh_aggs["gh_d48_std"].fillna(0)

def attach_lags(df):
    df = df.merge(lag_exact, on=["geohash", "minute"], how="left")
    df = df.merge(lag_hour,  on=["geohash", "hour"],   how="left")
    df = df.merge(gh_aggs,   on=["geohash"],           how="left")
    return df

train_fe = attach_lags(train_fe)
test_fe  = attach_lags(test_fe)

print(f"\nlag coverage:")
print(f"  train lag_same_ts non-null:   {train_fe['lag_same_ts'].notna().mean():.3f}")
print(f"  test  lag_same_ts non-null:   {test_fe['lag_same_ts'].notna().mean():.3f}")
print(f"  test  lag_same_hour non-null: {test_fe['lag_same_hour'].notna().mean():.3f}")
print(f"  test  gh_d48_mean non-null:   {test_fe['gh_d48_mean'].notna().mean():.3f}")

# ---------- self-rows of day-48 lag would leak for day-48 train ----------
# For day-48 training rows, the lag feature equals the row's own demand (mean over self).
# So this leaks. Solution: for day==48 training rows, set lag features to NaN.
day48_mask = train_fe["day"] == 48
for c in ["lag_same_ts", "lag_same_hour", "gh_d48_mean", "gh_d48_std",
          "gh_d48_med", "gh_d48_min", "gh_d48_max", "gh_d48_q25", "gh_d48_q75", "gh_d48_n"]:
    train_fe.loc[day48_mask, c] = np.nan

# ---------- day-49 leave-one-out aggregates (per-geohash mean of OTHER day-49 train rows) ----------
# Used as a feature for both day-49 train rows (LOO) and test rows (full mean over day-49 train).
d49 = train_fe[train_fe["day"] == 49]
d49_sum   = d49.groupby("geohash")[TARGET].sum()
d49_count = d49.groupby("geohash")[TARGET].count()
# LOO mean for day-49 train rows:
loo_mean = pd.Series(np.nan, index=train_fe.index)
gh_idx = train_fe["geohash"].values
y_idx  = train_fe[TARGET].values
mask49 = day48_mask.values == False  # day == 49
for i in np.where(mask49)[0]:
    g = gh_idx[i]
    n = d49_count.get(g, 0)
    if n > 1:
        loo_mean.iloc[i] = (d49_sum[g] - y_idx[i]) / (n - 1)
    elif n == 1:
        loo_mean.iloc[i] = np.nan  # only this one row
train_fe["gh_d49_mean_loo"] = loo_mean

test_d49_mean = (d49_sum / d49_count).reindex(test_fe["geohash"]).values
test_fe["gh_d49_mean_loo"] = test_d49_mean

# ---------- OOF target encoding for full geohash (smoothing) ----------
def target_encode_oof(train_df, test_df, col, y, n_splits=5, smoothing=20, seed=RNG):
    gm = y.mean()
    oof = np.zeros(len(train_df), dtype=np.float32)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in kf.split(train_df):
        agg = pd.DataFrame({col: train_df[col].iloc[tr].values, "y": y[tr]}) \
                .groupby(col)["y"].agg(["mean", "count"])
        sm = (agg["mean"]*agg["count"] + gm*smoothing) / (agg["count"] + smoothing)
        oof[va] = train_df[col].iloc[va].map(sm.to_dict()).fillna(gm).values
    agg = pd.DataFrame({col: train_df[col].values, "y": y}).groupby(col)["y"].agg(["mean","count"])
    sm = (agg["mean"]*agg["count"] + gm*smoothing) / (agg["count"] + smoothing)
    test_enc = test_df[col].map(sm.to_dict()).fillna(gm).values.astype(np.float32)
    return oof, test_enc

y_raw = train_fe[TARGET].astype(np.float32).values
for c in ["geohash"]:
    oof, te = target_encode_oof(train_fe, test_fe, c, y_raw)
    train_fe[f"{c}_te"] = oof
    test_fe[f"{c}_te"]  = te

# ---------- categorical encode ----------
cat_cols = ["gh3", "gh4", "gh5", "RoadType", "LargeVehicles", "Landmarks", "Weather"]
for c in cat_cols:
    train_fe[c] = train_fe[c].fillna("MISSING").astype(str)
    test_fe[c]  = test_fe[c].fillna("MISSING").astype(str)
temp_med = train_fe["Temperature"].median()
train_fe["Temperature"] = train_fe["Temperature"].fillna(temp_med)
test_fe["Temperature"]  = test_fe["Temperature"].fillna(temp_med)

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
all_cat = pd.concat([train_fe[cat_cols], test_fe[cat_cols]], axis=0)
enc.fit(all_cat)
train_fe[cat_cols] = enc.transform(train_fe[cat_cols])
test_fe[cat_cols]  = enc.transform(test_fe[cat_cols])

num_cols = ["day", "minute", "hour", "min_sin", "min_cos", "dow",
            "NumberofLanes", "Temperature"]
lag_cols = ["lag_same_ts", "lag_same_hour",
            "gh_d48_mean", "gh_d48_std", "gh_d48_med", "gh_d48_min", "gh_d48_max",
            "gh_d48_q25", "gh_d48_q75", "gh_d48_n",
            "gh_d49_mean_loo", "geohash_te"]
features = cat_cols + num_cols + lag_cols

X = train_fe[features].astype(np.float32)
y = y_raw
X_test = test_fe[features].astype(np.float32)
cat_idx = [features.index(c) for c in cat_cols]

print(f"\nfeatures: {len(features)}  cat: {len(cat_cols)}  num: {len(num_cols)}  lag: {len(lag_cols)}")
print(f"X={X.shape}  X_test={X_test.shape}")
print(f"NaNs in train features (per col):")
nan_counts = X.isna().sum()
print(nan_counts[nan_counts > 0])
print(f"setup elapsed: {time.time()-t0:.1f}s")

# ---------- CV ----------
def cv_score(model_factory, name, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RNG)
    scores, oof = [], np.zeros(len(y), dtype=np.float32)
    for fold, (tr, va) in enumerate(kf.split(X)):
        m = model_factory()
        m.fit(X.iloc[tr], y[tr])
        p = m.predict(X.iloc[va])
        oof[va] = p
        s = r2_score(y[va], p)
        scores.append(s)
        print(f"  [{name}] fold {fold+1}: RÂ²={s:.5f}", flush=True)
    mean, std = np.mean(scores), np.std(scores)
    print(f"  [{name}] mean RÂ²={mean:.5f} Â± {std:.5f}  (OOF RÂ²={r2_score(y, oof):.5f})", flush=True)
    # also report RÂ² restricted to day-49 rows (matches test distribution)
    day49 = (train_fe["day"] == 49).values
    print(f"  [{name}] day-49 only OOF RÂ²={r2_score(y[day49], oof[day49]):.5f}", flush=True)
    return mean, std, oof

print("\n=== HistGradientBoosting v2 (with lag) ===")
cv_score(lambda: HistGradientBoostingRegressor(
    max_iter=1500, learning_rate=0.03, max_leaf_nodes=255,
    min_samples_leaf=10, l2_regularization=1.0,
    categorical_features=cat_idx, random_state=RNG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=50),
    "hgb_v2", n_splits=5)

# ---------- final fit ----------
print("\n=== final HGB on full train ===", flush=True)
final = HistGradientBoostingRegressor(
    max_iter=2500, learning_rate=0.02, max_leaf_nodes=255,
    min_samples_leaf=10, l2_regularization=1.0,
    categorical_features=cat_idx, random_state=RNG,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=80)
final.fit(X, y)
print(f"  n_iter trained: {final.n_iter_}")

pred = np.clip(final.predict(X_test), 0, None)
sub = pd.DataFrame({"Index": test_fe["Index"].values, "demand": pred})
sub.to_csv(f"{DATA}/submission_v2.csv", index=False)
print(f"\nsubmission saved: {DATA}/submission_v2.csv  rows={len(sub)}")
print(sub.head())
print(f"\ntotal elapsed: {time.time()-t0:.1f}s")

train (77299, 11)  test (41778, 10)

lag coverage:
  train lag_same_ts non-null:   0.981
  test  lag_same_ts non-null:   0.889
  test  lag_same_hour non-null: 0.962
  test  gh_d48_mean non-null:   0.998

features: 27  cat: 7  num: 8  lag: 12
X=(77299, 27)  X_test=(41778, 27)
NaNs in train features (per col):
lag_same_ts        70876
lag_same_hour      69990
gh_d48_mean        69449
gh_d48_std         69449
gh_d48_med         69449
gh_d48_min         69449
gh_d48_max         69449
gh_d48_q25         69449
gh_d48_q75         69449
gh_d48_n           69449
gh_d49_mean_loo    69485
dtype: int64
setup elapsed: 1.4s

=== HistGradientBoosting v2 (with lag) ===
  [hgb_v2] fold 1: RÂ²=0.94636
  [hgb_v2] fold 2: RÂ²=0.94589
  [hgb_v2] fold 3: RÂ²=0.94493
  [hgb_v2] fold 4: RÂ²=0.94155
  [hgb_v2] fold 5: RÂ²=0.94554
  [hgb_v2] mean RÂ²=0.94485 Â± 0.00171  (OOF RÂ²=0.94490)
  [hgb_v2] day-49 only OOF RÂ²=0.95746

=== final HGB on full train ===
  n_iter trained: 692

submission saved: dataset\/sub